# What Makes a Good Shot? Expected Goals in Europe's Big Four Leagues, 2015/16
### DSE6311 Finalized Project Proposal

<img src="images/statsbomb_logo.png" width="210">

*Data provided by StatsBomb (StatsBomb Open Data, <https://github.com/hudl/open-data>).*

## Basics

| Item | Detail |
|---|---|
| **Project title** | What Makes a Good Shot? Expected Goals in Europe's Big Four Leagues, 2015/16 |
| **Team name** | Team Beta |
| **GitHub repository** | <https://github.com/aldengeorge26-droid/Soccer-Analytics-Capstone> |
| **Data source** | StatsBomb Open Data (credited with name and logo above) |

**Team members and standing roles**

| Member | Standing role |
|---|---|
| Brian Quevillon | Documentation|
| Alden George | Documentation |
| Brian D'Alelio | Communication |
| Anthony Dominici | ROLE |

**Weekly leads**

| Week | Roadmap milestone | Lead | Support |
|---|---|---|---|
| 4 | Polished EDA and unsupervised model | NAME | NAME |
| 5 | Baseline model | NAME | NAME |
| 6 | Additional models and tuning | NAME | NAME |
| 7 | Final model | NAME | NAME |
| 8 | Final report and presentation | NAME | NAME |

## Background & Project Question

Goals are rare in soccer, so a few deflections or good saves can change how a season looks. Clubs still judge teams on results, which can lead to bad decisions like firing a coach whose team creates good chances but fails to finish them (Brechot & Flepp, 2020). Expected goals (xG), the probability that a shot becomes a goal, separates chance creation from finishing and luck. xG predicts future success better than goals or shot totals (Mead et al., 2023), and Liverpool used similar analysis before hiring Jürgen Klopp (Schoenfeld, 2019). Distance and angle drive goal prediction (Rathke, 2017), and defender and goalkeeper positions improve location-only models (Lucey et al., 2015; Anzer & Bauer, 2021).

**Research question.** *Which characteristics of a shot (where it is taken, how the chance was created, and what the defense looks like at that moment) best predict its probability of becoming a goal?*

We aim for a well-calibrated xG model that shows coaching staff which chances are worth creating, without treating features as causes. This is hard because only about one in ten non-penalty shots is scored, the model must give realistic probabilities, and the same players and matches repeat, so splits must avoid leakage.

**Hypothesis.** A shot's chance of scoring depends on distance, angle, and where defenders and the goalkeeper are, because these decide how much of the goal the shooter can aim at. How the chance was created also matters. We expect that adding freeze-frame features, followed by build-up features, will lower log loss and Brier score compared with a location-only model when tested on later matches from the same season.

**Updates to the analysis plan.** We completed the chronological train/test split and set up separate leave-one-league-out splits using only the training matches. The test matches will be scored only once, after all tuning and model selection. We also compared three approaches to missing receipt values. Since their cross-validation results were nearly identical, we chose to replace missing values with zero and add a missing-value flag. We will compare feature groups in stages and calculate confidence intervals by sampling whole matches rather than individual shots, since shots from the same match are related.

## Data Preparation and Preprocessing

### Data Cleaning

**Scope and filtering.** The original StatsBomb Open Data contained 108,281 shots. We excluded shots from 274 matches without matching records and removed women's competitions, leaving 66,854 men's shots across 2,651 matches.

We checked which leagues had enough data for a full-season analysis. La Liga, the Premier League and Serie A each had all 380 matches from 2015/16, while Ligue 1 had 377. The other competitions were mostly tournaments, partial seasons or seasons covering one team. We kept these four leagues, giving us 37,888 shots across 1,517 matches. The three missing Ligue 1 matches make up only 0.2% of the expected total, so they should have little effect on our results.

We then removed 400 penalties because they are taken from a fixed position and have a much higher goal rate (75% versus 9.5%). We also removed 32 shots with no goalkeeper in the freeze frame. Of these, 19 were recorded as saves, showing that a goalkeeper was present but not captured. Seven direct corner shots were excluded because they were rare, with one recorded beyond the goal line. Finally, we removed 6 shots that had an assisting pass but no valid linked ball receipt. This left **37,443 shots with a 9.5% goal rate**.

**Missing values.** StatsBomb records True/False flags such as `under_pressure` and `shot_first_time` only when they are true, leaving them blank otherwise. We filled these missing values with False. About 28% of shots have missing receipt-to-shot values. Most had no assisting pass, while 15 had a valid receipt followed by another on-ball event before the shot. We kept these shots because their other features were still available and the missing values can reflect how the chance developed, such as a rebound or a shot after a dribble. The missing values are handled during preprocessing, as described in the Transformations section.

**Duplicates.** We checked for duplicate shots using StatsBomb's unique event `id`. No duplicate shot IDs or duplicate rows were found.

**Leakage fields.** We excluded from the model inputs any information that would not be available when the shot is taken, including the shot outcome, end location, deflections and final match score. The shot outcome was used to create our target, `is_goal`. StatsBomb's xG was kept only as an external benchmark, since its model may use different features and training data.

**Outliers.** We checked unusual shot distances, angles and defensive positions in `initial_eda_03`. No shots in the final dataset were recorded outside the pitch, and the calculated distances and angles matched their coordinates. Some extreme values represented real scoring opportunities. For example, 272 shots had the goalkeeper further from goal than the shooter, and 64% resulted in goals. We also kept 44 shots taken from the shooter's own half, none of which resulted in goals. These observations represent real game situations that our model should learn from.

### Transformations

**Encoding categorical variables.** Shot type, body part, technique and play pattern were one-hot encoded. Categories were learned from the training data only, so the test set could not influence which categories were included. True/False fields such as `under_pressure`, `shot_first_time` and `shot_one_on_one` were converted to 0 and 1.

**Scaling model inputs.** For logistic regression, we used `StandardScaler` to standardize the model inputs. The scaler is fitted on training data only and applied to the validation and test data using the same values.

**Collapsing categories.** Shot techniques appearing fewer than 150 times in the training data were grouped as `Rare or unseen`. This avoids creating separate predictors for techniques with very few shots and handles techniques not seen during training.

**Other transformations.** We calculated `shot_distance` and `shot_angle` using StatsBomb's 120 × 80 coordinate system. Both were checked against penalty shots and returned the expected 12 coordinate units and 36.87 degrees. We also used the freeze frames to count players in the shooting lane, estimate `percent_of_goal_open`, and measure goalkeeper positioning.

For shots following a pass, we calculated the time and straight-line distance between receiving the ball and shooting. Missing receipt measurements were filled with zero, with `receipt_measurements_missing` set to 1 to identify those shots. We compared this against median filling and dropping the features. All three performed nearly identically in cross-validation (log loss 0.2506–0.2507 and Brier score 0.0712), so we kept zero filling with the missing flag. Appendix A contains the full variable list.

The transformations that learn from data are fitted using training shots only and refitted within each cross-validation fold. This prevents validation data from influencing preprocessing. After encoding, the model inputs contain 37 predictors.